# 01 — Data Exploration

Looking at what the Silver lap data actually contains: distributions, quality flags, session coverage, and what 'clean racing lap' means in practice.

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pathlib
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.facecolor'] = '#080c14'
matplotlib.rcParams['axes.facecolor']   = '#0f172a'
matplotlib.rcParams['axes.edgecolor']   = '#1e293b'
matplotlib.rcParams['xtick.color']      = '#8b9bb4'
matplotlib.rcParams['ytick.color']      = '#8b9bb4'
matplotlib.rcParams['text.color']       = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor']  = '#8b9bb4'
matplotlib.rcParams['figure.dpi']       = 120

In [ ]:
# Load all silver laps
SILVER = pathlib.Path('../data/silver/laps')
files  = sorted(SILVER.rglob('*.parquet'))
print(f'{len(files)} session files')

df = pl.concat([pl.read_parquet(f) for f in files])
print(f'Total rows: {len(df):,}')
print(f'Columns ({len(df.columns)}): {df.columns}')

In [ ]:
# How many sessions, drivers, laps?
print('Sessions:', df['session_id'].n_unique())
print('Drivers: ', df['driver_id'].n_unique())
print('Rows per session (median):', int(df.group_by('session_id').len()['len'].median()))
df.group_by('session_id').len().sort('len').tail(5)

In [ ]:
# The clean-lap filter — this is why raw MAE is terrible
raw   = df.filter(pl.col('lap_time_s').is_not_null())
clean = df.filter(pl.col('is_valid_training_lap') == True)
print(f'Raw laps:   {len(raw):,}  mean={raw["lap_time_s"].mean():.2f}s  std={raw["lap_time_s"].std():.2f}s')
print(f'Clean laps: {len(clean):,}  mean={clean["lap_time_s"].mean():.2f}s  std={clean["lap_time_s"].std():.2f}s')
print(f'Retention:  {len(clean)/len(raw):.1%}')

In [ ]:
# What gets filtered out?
df = df.filter(pl.col('lap_time_s').is_not_null())
pit_in  = df.filter(pl.col('is_pit_in')  == True)
pit_out = df.filter(pl.col('is_pit_out') == True)
print(f'Pit-in laps:  {len(pit_in):,}')
print(f'Pit-out laps: {len(pit_out):,}')

# Show how pit-out laps inflate lap times
print(f'\nNormal lap time: {clean["lap_time_s"].median():.2f}s')
print(f'Pit-out laps:    {pit_out["lap_time_s"].drop_nulls().median():.2f}s  (+{pit_out["lap_time_s"].drop_nulls().median() - clean["lap_time_s"].median():.1f}s)')

In [ ]:
# Lap time distributions by compound
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {'SOFT': '#ff1801', 'MEDIUM': '#f59e0b', 'HARD': '#94a3b8'}
for ax, compound in zip(axes, ['SOFT', 'MEDIUM', 'HARD']):
    data = clean.filter(pl.col('compound') == compound)['lap_time_s'].drop_nulls().to_numpy()
    if len(data):
        ax.hist(data, bins=40, color=colors[compound], alpha=0.85, edgecolor='#080c14', linewidth=0.3)
        ax.axvline(np.median(data), color='white', linewidth=1.5, linestyle='--', label=f'Median: {np.median(data):.2f}s')
        ax.set_title(f'{compound}  (n={len(data):,})', fontsize=11)
        ax.set_xlabel('Lap time (s)')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
plt.suptitle('Clean lap time distributions by compound', fontsize=13, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

In [ ]:
# Lap count by compound
clean.group_by('compound').agg(pl.len().alias('count')).sort('count', descending=True)

In [ ]:
# Hard compound is severely underrepresented — root cause of high Hard MAE
total = len(clean)
for row in clean.group_by('compound').agg(pl.len().alias('n')).sort('n', descending=True).iter_rows(named=True):
    print(f'{row["compound"]:8s}: {row["n"]:5d} laps  ({row["n"]/total:.1%})')

In [ ]:
# Tyre degradation over stint laps
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, compound in zip(axes, ['SOFT', 'MEDIUM', 'HARD']):
    subset = (clean
        .filter(pl.col('compound') == compound)
        .filter(pl.col('tyre_age') <= 40)
        .group_by('tyre_age')
        .agg(pl.col('lap_time_s').median().alias('median_lap'))
        .sort('tyre_age')
    )
    if len(subset) > 3:
        ax.scatter(subset['tyre_age'].to_numpy(), subset['median_lap'].to_numpy(),
                   color=colors[compound], s=25, alpha=0.8)
        ax.set_title(f'{compound} — lap time vs tyre age', fontsize=10)
        ax.set_xlabel('Tyre age (laps)')
        ax.set_ylabel('Median lap time (s)')
plt.suptitle('Tyre degradation curves', fontsize=13, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

In [ ]:
# Session coverage — which years/races are in the dataset?
clean.select('session_id').unique().sort('session_id')